# Smart Bin — YOLOv8 retraining (RealWaste, 4 classes)

**Project:** AI-based Smart Bin, BS Final Year Project, IUB — Plastic / Paper / Food / Others, conveyor + ESP32-CAM + Arduino Uno.

**What this notebook does:** retrains YOLOv8 **classification** (`yolov8n-cls`) on the remapped RealWaste dataset and produces every number the IEEE paper needs — dataset counts, train/val/test split, top-1 / top-5 accuracy, per-class precision / recall / F1, confusion matrix, sample predictions with images, and single-image latency.

**Important — classification, not detection:** the dataset is one folder of images per class, with no bounding boxes. That trains a *classifier*, so the correct metrics are accuracy / precision / recall / F1 and a confusion matrix — **not mAP**. Do not report mAP for this model. The FYP report's training graphs (top-1 / top-5 curves over ~30 epochs) are also classification curves, which matches this notebook.

**Dataset (author-supplied, to verify in the count cell below):** Kaggle RealWaste (`joebeachcapital/realwaste`, Single et al. 2023), remapped 9 classes → 4 classes. Author-supplied totals: Paper 1,831 · Plastic 1,674 · Food 1,461 · Others 1,885 = **6,851**. ⚠️ The original RealWaste has **4,752** images (Cardboard 461, Food Organics 411, Glass 420, Metal 790, Misc. Trash 495, Paper 500, Plastic 921, Textile 318, Vegetation 436), so 6,851 can only be right if local phone images and/or augmentation were added — the count cell below prints both tables and flags any mismatch. Do **not** write 6,851 into the paper until the count cell confirms it from the actual folders.

**Expected folder layout** (put your data here before running):
```
realwaste/
  paper/    *.jpg
  plastic/  *.jpg
  food/     *.jpg
  others/   *.jpg
```
Local phone images go **inside the same class folders** (e.g. your 71 Food phone photos inside `food/`), so they are counted and split with the rest. Run in Google Colab (GPU) or locally with a GPU if you have one; CPU training will be very slow.

## 1 — Setup

In [ ]:
!pip install -q ultralytics scikit-learn matplotlib seaborn
import sys, torch, ultralytics, sklearn
print('python', sys.version)
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('ultralytics', ultralytics.__version__)
print('sklearn', sklearn.__version__)

## 2 — Config (edit only this cell)

In [ ]:
# --- edit these paths / settings ---
DATA_DIR = 'realwaste'          # folder with paper/ plastic/ food/ others/ inside
SPLIT_DIR = 'realwaste_split'   # stratified train/val/test copy created below
CLASSES = ['paper', 'plastic', 'food', 'others']
SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
MODEL = 'yolov8n-cls.pt'        # start here; repeat with 'yolov8s-cls.pt' for the ablation
EPOCHS = 30                     # matches the FYP training curves (~30 epochs)
IMGSZ = 224                     # YOLOv8-cls default; try 320 only if GPU memory allows
BATCH = 32
PROJECT = 'smartbin_runs'       # runs/ folder for weights, results.csv, plots
RUN_NAME = 'yolov8n_cls_4class'

# Author-supplied totals from the FYP team — the count cell checks these against disk.
EXPECTED = {'paper': 1831, 'plastic': 1674, 'food': 1461, 'others': 1885}
# Original RealWaste (Single et al. 2023, Kaggle/UCI) — for the remap table in the paper.
ORIGINAL = {'cardboard': 461, 'food organics': 411, 'glass': 420, 'metal': 790,
            'miscellaneous trash': 495, 'paper': 500, 'plastic': 921,
            'textile trash': 318, 'vegetation': 436}  # total 4,752

## 3 — Count the dataset and check it against the claimed totals

Run this first. If any class count differs from `EXPECTED`, stop and fix the folders (or tell Muse the real counts) — do not train on a silently different dataset.

In [ ]:
from pathlib import Path
from collections import Counter

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
base = Path(DATA_DIR)
counts = {}
for cls in CLASSES:
    files = [p for p in (base / cls).rglob('*') if p.suffix.lower() in IMG_EXTS]
    counts[cls] = len(files)
    print(f'{cls:10s} {len(files):5d}   expected {EXPECTED[cls]:5d}   diff {len(files) - EXPECTED[cls]:+d}')
total = sum(counts.values())
print(f"{'TOTAL':10s} {total:5d}   expected {sum(EXPECTED.values()):5d}   diff {total - sum(EXPECTED.values()):+d}")
print('Original RealWaste total:', sum(ORIGINAL.values()))
print('Remap used: paper->paper; plastic->plastic; food organics->food;')
print('  cardboard+glass+metal+misc trash+textile+vegetation -> others')
print('  (original others classes sum to', 461+420+790+495+318+436, '; check why your others =', counts.get('others'), ')')
assert all(counts[c] > 0 for c in CLASSES), 'A class folder is empty or missing!'

## 4 — Stratified train / val / test split (70 / 15 / 15, by image, seed 42)

Split is stratified per class, so every class keeps the same proportions in train, val and test. The test set is held out and is **never** used for training or for picking the confidence threshold. Splitting by image is acceptable here because each image is a different waste item; if the same physical item was photographed several times (e.g. phone bursts), group those photos into the same split by hand first.

In [ ]:
import random, shutil
from pathlib import Path

random.seed(SEED)
out = Path(SPLIT_DIR)
if out.exists():
    shutil.rmtree(out)
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        (out / split / cls).mkdir(parents=True, exist_ok=True)
for cls in CLASSES:
    files = [p for p in (Path(DATA_DIR) / cls).rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
    random.shuffle(files)
    n = len(files)
    n_train, n_val = int(n * TRAIN_FRAC), int(n * VAL_FRAC)
    for split, subset in [('train', files[:n_train]), ('val', files[n_train:n_train+n_val]), ('test', files[n_train+n_val:])]:
        for p in subset:
            shutil.copy2(p, out / split / cls / p.name)
    print(f'{cls:10s} total {n:5d} -> train {n_train:5d}  val {n_val:5d}  test {n - n_train - n_val:5d}')
print('Split written to', SPLIT_DIR)

## 5 — Train YOLOv8 classification

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)  # yolov8n-cls, pretrained on ImageNet
results = model.train(
    data=SPLIT_DIR,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=10,          # early stopping; raise if the run stops before epoch 30
    seed=SEED,
    project=PROJECT,
    name=RUN_NAME,
    plots=True,           # saves loss / accuracy curves like the FYP Fig. 6.2
    save=True,
    verbose=True,
)
print('Best weights:', results.save_dir / 'weights' / 'best.pt')

## 6 — Validate (top-1 / top-5 accuracy) and test (held-out test set)

In [ ]:
from pathlib import Path
best = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
model = YOLO(str(best))
print('--- validation set ---')
val = model.val(data=SPLIT_DIR, split='val')
print('--- held-out test set ---')
test = model.val(data=SPLIT_DIR, split='test')
print('val top1:', round(val.top1, 4), '| val top5:', round(val.top5, 4))
print('test top1:', round(test.top1, 4), '| test top5:', round(test.top5, 4))

## 7 — Per-class precision / recall / F1 + confusion matrix (the paper's Table: Results)

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

test_dir = Path(SPLIT_DIR) / 'test'
y_true, y_pred, confs = [], [], []
for i, cls in enumerate(CLASSES):
    for p in sorted((test_dir / cls).glob('*')):
        if p.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:
            continue
        r = model.predict(str(p), verbose=False)[0]
        y_true.append(i); y_pred.append(int(r.probs.top1)); confs.append(float(r.probs.top1conf))
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
Path('paper_metrics.json').write_text(json.dumps({
    'model': MODEL, 'epochs': EPOCHS, 'imgsz': IMGSZ, 'seed': SEED,
    'split': {'train': TRAIN_FRAC, 'val': VAL_FRAC, 'test': TEST_FRAC},
    'counts': {c: len(list((Path(DATA_DIR)/c).rglob('*'))) for c in CLASSES},
    'test_n': len(y_true),
    'classification_report': report,
}, indent=2))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix — held-out test set')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=200); plt.show()
print('Saved: paper_metrics.json, confusion_matrix.png')

## 8 — Sample predictions with images (paper figure: correct + wrong cases)

In [ ]:
import random, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

random.seed(SEED)
test_dir = Path(SPLIT_DIR) / 'test'
samples = []
for cls in CLASSES:
    files = sorted((test_dir / cls).glob('*'))
    samples += random.sample(files, min(3, len(files)))
fig, axes = plt.subplots(4, 3, figsize=(10, 12))
for ax, p in zip(axes.ravel(), samples):
    r = model.predict(str(p), verbose=False)[0]
    pred = CLASSES[int(r.probs.top1)]; conf = float(r.probs.top1conf)
    ax.imshow(Image.open(p)); ax.axis('off')
    ok = 'OK' if pred == p.parent.name else 'WRONG'
    ax.set_title(f"true: {p.parent.name} | pred: {pred} ({conf:.2f}) {ok}", fontsize=9)
plt.tight_layout(); plt.savefig('sample_predictions.png', dpi=200); plt.show()
print('Saved: sample_predictions.png — send this + confusion_matrix.png + paper_metrics.json to Muse')

## 9 — Confidence-threshold sweep for the Others / reject route (paper ablation)

The bin routes an item to Others when confidence is below a threshold. This cell shows the trade-off: a higher threshold sends more items to Others (higher routed precision, lower coverage). Pick the threshold on the **validation** set, report it, and apply it unchanged to the test set.

In [ ]:
import numpy as np
from pathlib import Path

# reuse y_true / y_pred / confs from section 7; if rerun alone, recompute them there first
y_true = np.array(y_true); y_pred = np.array(y_pred); confs = np.array(confs)
print('threshold | routed | routed precision | coverage')
for tau in [0.5, 0.6, 0.7, 0.8, 0.9]:
    routed = confs >= tau
    prec = (y_pred[routed] == y_true[routed]).mean() if routed.any() else float('nan')
    print(f'{tau:.1f}       | {routed.sum():4d}   | {prec:.4f}           | {routed.mean():.3f}')
print('FYP report used 0.60 as the confidence threshold. Compare it with the sweep above.')

## 10 — Single-image latency (for the paper's latency table)

Run on the same PC that will run the desktop application. Report mean and p95, with hardware + imgsz + model variant stated — a single 'real-time' claim is not enough.

In [ ]:
import time, numpy as np
from pathlib import Path

test_files = list((Path(SPLIT_DIR) / 'test').rglob('*.jpg'))[:50]
times = []
for p in test_files:
    t0 = time.perf_counter()
    _ = model.predict(str(p), verbose=False)
    times.append((time.perf_counter() - t0) * 1000)
times = np.array(times)
print(f"n={len(times)}  mean {times.mean():.1f} ms  median {np.median(times):.1f} ms  p95 {np.percentile(times, 95):.1f} ms  max {times.max():.1f} ms")
print('Note: this is model inference only. Add ESP32-CAM capture + Wi-Fi + serial + servo time')
print('from the desktop app log for the end-to-end latency (trigger -> sorted item).')

## 11 — What to send back for the paper

Send Muse these files (they are in this notebook's folder / `smartbin_runs/`):

1. `paper_metrics.json` — test accuracy + per-class precision / recall / F1
2. `confusion_matrix.png` and `sample_predictions.png` — paper figures
3. `smartbin_runs/yolov8n_cls_4class/results.csv` — training curves (loss, top-1/top-5 per epoch)
4. `smartbin_runs/yolov8n_cls_4class/weights/best.pt` — keep this, do not send the binary in chat; just confirm its path
5. The count-cell output (Section 3) — the verified per-class totals, and how many of them are local phone photos per class
6. Latency output from Section 10 + the PC's CPU/GPU model

Also confirm: (a) are the Section-3 counts before or after augmentation? (b) how many local phone images per class, not just Food's 71? (c) which classes did you drop or subsample when merging the other six RealWaste classes into Others (originally 2,920 images)?

**References for the paper:** S. Single, S. Iranmanesh & R. Raad, “RealWaste: A Novel Real-Life Data Set for Landfill Waste Classification Using Deep Learning,” *Information*, vol. 14, no. 12, art. 633, 2023, doi: 10.3390/info14120633. Dataset: Kaggle `joebeachcapital/realwaste`, also UCI Machine Learning Repository (DOI 10.24432/C5SS4G), CC BY-NC-SA 4.0 (non-commercial research/educational use). YOLOv8: Ultralytics, `yolov8n-cls`, software/docs — cite as software, it has no peer-reviewed paper.